Ingest documents and chunk

In [12]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

ROOT_DIR = Path.cwd()
AV_PAPER_DIR = ROOT_DIR / "data" / "av_paper.pdf"
av_paper = PyPDFLoader(str(AV_PAPER_DIR)).load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(av_paper)
print(f"split into {len(chunks)} chunks")

split into 146 chunks


In [18]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load environment variables from .env file
load_dotenv()

# 1. Load and Chunk
loader = PyPDFLoader("data/av_paper.pdf")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

# 2. Initialize Embeddings (reads GOOGLE_API_KEY from env automatically)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    output_dimensionality=768  # Set to match your Pinecone index dimension
)

# 3. Store in Pinecone (reads PINECONE_API_KEY from env automatically)
index_name = "av-paper-index"

vector_store = PineconeVectorStore.from_documents(
    documents=chunks, embedding=embeddings, index_name=index_name
)

# 4. Query
query = "What are the main findings in the paper?"
results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} (Page {doc.metadata.get('page', 'N/A')}) ---")
    print(doc.page_content)
    print("\n")

--- Result 1 (Page 0.0) ---
and
 
Cybersecurity
...........................................................................................................
5
 
2.7
 
Selection
 
Criteria
 
for
 
Reviewed
 
Literature
........................................................................................................
5
 
3.
 
Research
 
Methodology
..............................................................................................................................................
6
 
4.
 
Experimental
 
Testing
................................................................................................................................................
6
 
5.
 
Results:
 
Error
 
and
 
Reliability
..................................................................................................................................
6
 
5.1
 
Sensor
 
Degradation
 
Patterns
...........................................................................................................